In [ ]:
!pip install -q nlp-id
# nlp-id pins an old huggingface_hub that breaks recent transformers -> upgrade it back
!pip install -q -U huggingface_hub
# If Colab asks to restart the session after this cell, restart and run all again.

In [ ]:
import os
os.environ['NLTK_ALLOW_PROXIED_URLOPEN'] = '1'  # newer NLTK blocks downloads through Colab's proxy without this
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
from nltk.tokenize import word_tokenize, sent_tokenize

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────
# Put scores.pkl, question_ids.pkl, preprocessed_texts.pkl and
# preprocessed_sbert_texts.pkl in DATA_DIR (Google Drive when using Colab).
DATA_DIR    = "/content/drive/MyDrive/aes_data/"
ESSAY_PATH  = DATA_DIR
SCORE_PATH  = DATA_DIR
OUTPUT_PATH = "/content/drive/MyDrive/aes_results/"

In [ ]:
"""
=============================================================
  AES — Paradigma A: IndoSBERT-large Fine-tuning (BERT ONLY)
  Dataset  : preprocessed_texts.pkl, scores.pkl
  Model    : denaya/indoSBERT-large
  Arsitektur:
    (0) Transformer: BertModel LARGE — 24 layer, hidden 1024
    (1) Pooling    : mean pooling (pooling_mode_mean_tokens=True)
    (2) Dense      : Linear(1024→256) + Tanh  → output dim 256
  FC head   : Linear(256 → 128) + ReLU + Dropout + Linear(128 → 1)
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation
=============================================================
"""

import os, time, traceback, random
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pynvml
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error


# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    "essay"        : f"{ESSAY_PATH}preprocessed_sbert_texts.pkl",
    "score"        : f"{SCORE_PATH}scores.pkl",
    "question_ids" : f"{SCORE_PATH}question_ids.pkl",
    "model_name"      : "denaya/indoSBERT-large",
    "max_len"         : 256,
    "sbert_output_dim": 256,
    "lr_encoder"      : 2e-5,
    "lr_head"         : 1e-3,
    "weight_decay"    : 0.01,
    "warmup_steps"    : 100,
    "max_epochs"      : 30,
    "batch_size"      : 64,
    "patience"        : 5,
    "dropout"         : 0.1,
    "n_folds"         : 10,
    "seed"            : 42,
    "output_dir"      : f"{OUTPUT_PATH}output_indosbert_ft_no_fe",
    "score_min"       : 2,
    "score_max"       : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device           : {DEVICE}")
print(f"Model            : {CFG['model_name']}")
print(f"Backbone         : BERT-large (24L, hidden 1024)")
print(f"Output embedding : {CFG['sbert_output_dim']} dim (setelah Dense Tanh)")
print(f"Fitur tambahan   : TIDAK DIGUNAKAN (BERT only)")


# ══════════════════════════════════════════════════════════
#  REPRODUCIBILITY
# ══════════════════════════════════════════════════════════
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating, max_rating):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)
    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1
    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored
    weights    = np.array([[(i - j)**2 / (num_ratings - 1)**2
                             for j in range(num_ratings)]
                            for i in range(num_ratings)])
    denom = np.sum(weights * expected)
    return 0.0 if denom == 0 else 1.0 - np.sum(weights * conf_mat) / denom


# ══════════════════════════════════════════════════════════
#  DATASET (tokenisasi dilakukan di __getitem__)
# ══════════════════════════════════════════════════════════
class EssayDataset(Dataset):
    def __init__(self, texts, scores, tokenizer, max_len):
        self.texts     = texts
        self.scores    = scores
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        # Tokenisasi di sini, mirip dengan kode IndoBERT
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids"      : enc["input_ids"].squeeze(0),
            "attention_mask" : enc["attention_mask"].squeeze(0),
            "score"          : torch.tensor(self.scores[idx], dtype=torch.float32),
        }


# ══════════════════════════════════════════════════════════
#  MODEL — IndoSBERTRegressor (BERT only, tanpa fitur tambahan)
# ══════════════════════════════════════════════════════════
class IndoSBERTRegressor(nn.Module):
    def __init__(self, model_name, device, dropout=0.1):
        super().__init__()
        self.sbert   = SentenceTransformer(model_name, device=str(device))
        emb_dim      = CFG["sbert_output_dim"]   # 256
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(emb_dim, 128)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(128, 1)

    def forward(self, input_ids, attention_mask):
        """
        Forward manual melalui pipeline SBERT dengan gradient aktif.
        Sekarang menerima input_ids dan attention_mask yang sudah ditokenisasi,
        sehingga konsisten dengan dataset.
        """
        # Modul 0: Transformer BERT-large
        transformer_out = self.sbert[0].auto_model(
            input_ids=input_ids, attention_mask=attention_mask)
        token_emb = transformer_out.last_hidden_state   # (B, seq_len, 1024)

        # Modul 1: Mean pooling
        mask     = attention_mask.unsqueeze(-1).float()
        sum_emb  = torch.sum(token_emb * mask, dim=1)
        sum_mask = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_emb = sum_emb / sum_mask                  # (B, 1024)

        # Modul 2: Dense Linear(1024→256) + Tanh
        out = self.sbert[2]({"sentence_embedding": mean_emb})
        emb = out["sentence_embedding"]               # (B, 256)

        emb = self.dropout(emb)
        h   = self.dropout(self.relu(self.fc1(emb)))
        return self.fc2(h).squeeze(-1)


# ══════════════════════════════════════════════════════════
#  TRAINING & EVALUATION
# ══════════════════════════════════════════════════════════
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        preds = model(batch["input_ids"].to(device),
                      batch["attention_mask"].to(device))
        loss  = criterion(preds, batch["score"].to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device, score_min, score_max):
    model.eval()
    all_preds, all_true, total_loss = [], [], 0.0
    criterion = nn.MSELoss()
    with torch.no_grad():
        for batch in loader:
            preds = model(batch["input_ids"].to(device),
                          batch["attention_mask"].to(device))
            total_loss += criterion(preds, batch["score"].to(device)).item()
            all_preds.extend(preds.cpu().numpy().tolist())
            all_true.extend(batch["score"].cpu().numpy().tolist())
    all_preds = np.array(all_preds)
    all_true  = np.array(all_true)
    return (total_loss / len(loader),
            mean_squared_error(all_true, all_preds),
            quadratic_weighted_kappa(all_true, all_preds, score_min, score_max),
            all_preds, all_true)

# ══════════════════════════════════════════════════════════
#  RUN FOLD (dengan pembersihan memori)
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx, texts, scores, question_ids, tokenizer, cfg, device, es_idx=None):

    print(f"\n{'='*55}\n  FOLD {fold_idx+1}/{cfg['n_folds']}\n{'='*55}")
    set_seed(cfg["seed"] + fold_idx)
    if device.type == "cuda":  # [FIX-MEM] true per-fold training peak
        import gc; gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(device)  # [FIX-LEAK]

    # Bersihkan cache sebelum memulai fold (jika ada sisa dari fold sebelumnya)
    if device.type == "cuda":
        torch.cuda.empty_cache()

    y_train = scores[train_idx]
    y_val   = scores[val_idx]

    train_ds = EssayDataset([texts[i] for i in train_idx], y_train,
                             tokenizer, cfg["max_len"])
    val_ds   = EssayDataset([texts[i] for i in val_idx],   y_val,
                             tokenizer, cfg["max_len"])
    train_dl = DataLoader(train_ds, batch_size=cfg["batch_size"],
                          shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                          shuffle=False, num_workers=0)
    # [FIX-LEAKAGE] early-stopping set (NOT the test fold)
    y_es     = scores[es_idx]
    es_ds   = EssayDataset([texts[i] for i in es_idx],   y_es,
                             tokenizer, cfg["max_len"])
    es_dl   = DataLoader(es_ds,   batch_size=cfg["batch_size"],
                          shuffle=False, num_workers=0)

    model = IndoSBERTRegressor(cfg["model_name"], device, cfg["dropout"]).to(device)

    pipeline_params = (list(model.sbert[0].parameters()) +
                       list(model.sbert[2].parameters()))
    head_params     = (list(model.fc1.parameters()) +
                       list(model.fc2.parameters()))

    optimizer = torch.optim.AdamW([
        {"params": pipeline_params, "lr": cfg["lr_encoder"]},
        {"params": head_params,     "lr": cfg["lr_head"]},
    ], weight_decay=cfg["weight_decay"])

    total_steps = len(train_dl) * cfg["max_epochs"]
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=cfg["warmup_steps"],
        num_training_steps=total_steps)
    criterion = nn.MSELoss()

    best_val_loss, best_model_state, patience_counter = float("inf"), None, 0
    train_loss_history, val_loss_history = [], []
    val_qwk_history,   val_mse_history   = [], []
    n_epochs_actual = 0

    t_start = time.perf_counter()
    for epoch in range(cfg["max_epochs"]):
        n_epochs_actual = epoch + 1
        train_loss = train_one_epoch(model, train_dl, optimizer,
                                     scheduler, criterion, device)
        val_loss, val_mse, val_qwk, _, _ = evaluate(
            model, es_dl, device, cfg["score_min"], cfg["score_max"])

        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)
        val_qwk_history.append(val_qwk)
        val_mse_history.append(val_mse)

        print(f"  Epoch {epoch+1:02d}/{cfg['max_epochs']} | "
              f"train={train_loss:.4f} | val={val_loss:.4f} | "
              f"QWK={val_qwk:.4f} | MSE={val_mse:.4f}")

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg["patience"]:
                print(f"  Early stopping pada epoch {epoch+1}")
                break

    training_time = time.perf_counter() - t_start

    model.load_state_dict(best_model_state)

    # --- Inference time ---
    model.eval()
    sample_batch = next(iter(val_dl))
    N_INFER = min(100, len(val_ds))
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N_INFER):
            model(sample_batch["input_ids"][:1].to(device),
                  sample_batch["attention_mask"][:1].to(device))
    inference_ms = (time.perf_counter() - t0) / N_INFER * 1000

    # --- Memory tracking via pynvml ---
    if device.type == "cuda":
        pynvml.nvmlInit()
        handle   = pynvml.nvmlDeviceGetHandleByIndex(device.index or 0)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        mem_mb   = torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # [FIX-MEM] was whole-GPU "used"
        mem_device = "cuda"
        pynvml.nvmlShutdown()
    else:
        mem_mb     = 0.0
        mem_device = "cpu (not tracked)"

    _, final_mse, final_qwk, preds, true = evaluate(
        model, val_dl, device, cfg["score_min"], cfg["score_max"])

    fold_preds_df = pd.DataFrame({
        "sample_idx"  : val_idx,
        "question_id" : question_ids[val_idx],
        "true_score"  : true,
        "pred_score"  : preds,
        "residual"    : preds - true,
        "abs_error"   : np.abs(preds - true),
        "fold"        : fold_idx + 1,
    })

    print(f"\n  → Fold {fold_idx+1}: QWK={final_qwk:.4f} | MSE={final_mse:.4f} | "
          f"time={training_time:.1f}s | inf={inference_ms:.2f}ms | mem={mem_mb:.1f}MB")

    # ─── PEMBERSIHAN MEMORI ──────────────────────────────
    del model
    del optimizer
    del scheduler
    del train_dl, val_dl
    del train_ds, val_ds
    if device.type == "cuda":
        torch.cuda.empty_cache()
    # ──────────────────────────────────────────────────────

    return {
        "fold"            : fold_idx + 1,
        "qwk"             : final_qwk,
        "mse"             : final_mse,
        "training_time_s" : training_time,
        "inference_ms"    : inference_ms,
        "memory_mb"       : mem_mb,
        "memory_device"   : mem_device,
        "best_val_loss"   : best_val_loss,
        "n_epochs_actual" : n_epochs_actual,
        "val_idx"         : val_idx.tolist(),
        "train_idx"       : train_idx.tolist(),
        "history": {
            "train_loss": train_loss_history,
            "val_loss"  : val_loss_history,
            "val_qwk"   : val_qwk_history,
            "val_mse"   : val_mse_history,
        },
    }, fold_preds_df


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    set_seed(CFG["seed"])

    print("\n[1/4] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])

    if os.path.exists(CFG["question_ids"]):
        question_ids = joblib.load(CFG["question_ids"])
    else:
        n_per_q = len(texts) // 10
        question_ids = np.repeat(np.arange(1, 11), n_per_q).astype(int)
        if len(question_ids) < len(texts):
            question_ids = np.append(question_ids,
                np.full(len(texts) - len(question_ids), 10, dtype=int))
        joblib.dump(question_ids, CFG["question_ids"])
        print(f"  question_ids dibuat dan disimpan (asumsi {n_per_q} esai/soal)")

    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())
    print(f"  Sampel: {len(texts)} | Skor: {CFG['score_min']}–{CFG['score_max']} | "
          f"Mean: {scores.mean():.2f}±{scores.std():.2f}")
    print(f"  PENTING: do_lower_case=False — teks tidak di-lowercase")

    print(f"\n[2/4] Mengambil tokenizer dari model {CFG['model_name']}...")
    sbert_temp = SentenceTransformer(CFG["model_name"], device=str(DEVICE))
    tokenizer  = sbert_temp.tokenizer
    del sbert_temp   # tidak perlu lagi

    print(f"\n[3/4] Memulai {CFG['n_folds']}-fold CV (tanpa fitur linguistik)...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
    all_results, all_preds_dfs = [], []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(texts)):

        train_idx, es_idx = inner_split(train_idx, scores, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df = run_fold(
                fold_idx, train_idx, val_idx,
                texts, scores, question_ids,
                tokenizer, CFG, DEVICE, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)

            # Pembersihan tambahan di tingkat loop
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1}: {e}")
            traceback.print_exc()
            # Bersihkan juga jika terjadi error
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    print(f"\n[4/4] Menyimpan hasil...")
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"])
    overall_mse = mean_squared_error(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values)

    score_dist = {}
    for q in range(1, 11):
        mask = question_ids == q
        if mask.sum() > 0:
            score_dist[int(q)] = {
                "mean"  : float(scores[mask].mean()),
                "std"   : float(scores[mask].std()),
                "n"     : int(mask.sum()),
                "counts": {int(k): int(v) for k, v in
                           zip(*np.unique(scores[mask].astype(int),
                                          return_counts=True))},
            }

    experiment_data = {
        "config"            : CFG,
        "fold_results"      : all_results,
        "fold_predictions"  : all_preds_df,
        "overall_qwk"       : overall_qwk,
        "overall_mse"       : overall_mse,
        "scores"            : scores,
        "question_ids"      : question_ids,
        "score_distribution": score_dist,
        "model_name"        : CFG["model_name"],
        "paradigm"          : "fine-tuning (BERT only, no features)",
        "n_samples"         : len(texts),
        "timestamp"         : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    joblib_path = os.path.join(CFG["output_dir"], "experiment_results_indosbert_ft_bert_only.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df     = pd.DataFrame(all_results)
    results_df_csv = results_df.drop(columns=["history","val_idx","train_idx"],
                                      errors="ignore")
    results_df_csv.to_csv(
        os.path.join(CFG["output_dir"], "cv_results_indosbert_ft_bert_only.csv"), index=False)
    all_preds_df.to_csv(
        os.path.join(CFG["output_dir"], "all_predictions_indosbert_ft_bert_only.csv"), index=False)

    print(f"\n{'='*55}")
    print("  RINGKASAN — IndoSBERT-large Fine-tuning (BERT ONLY)")
    print(f"{'='*55}")
    print(results_df_csv.to_string(index=False))
    print(f"\n  Rata-rata ± Std:")
    for m in ["qwk","mse","training_time_s","inference_ms","memory_mb"]:
        print(f"    {m:<20s}: {results_df[m].mean():.4f} ± {results_df[m].std():.4f}")
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")
    print(f"\n  Disimpan di     : {joblib_path}")

    return experiment_data


if __name__ == "__main__":
    main()

---


In [ ]:
"""
=============================================================
  AES — Paradigma A: IndoBERT Fine-tuning (BERT ONLY, tanpa fitur linguistik)
  Dataset  : preprocessed_texts.pkl, scores.pkl   (SAMA dengan versi +FE)
  Model    : indobenchmark/indobert-base-p2
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation  (SAMA dengan versi +FE, untuk ablasi yang valid)
=============================================================
"""

import os, time, traceback, random
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pynvml
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error


# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    "essay"        : f"{ESSAY_PATH}preprocessed_texts.pkl",
    "score"        : f"{SCORE_PATH}scores.pkl",
    "question_ids" : f"{SCORE_PATH}question_ids.pkl",
    "model_name"   : "indobenchmark/indobert-base-p2",
    "max_len"      : 512,
    "lr_encoder"   : 2e-5,
    "lr_head"      : 1e-3,
    "weight_decay" : 0.01,
    "warmup_ratio" : 0.10,
    "max_epochs"   : 30,
    "batch_size"   : 64,
    "patience"     : 5,
    "dropout"      : 0.1,
    "n_folds"      : 10,
    "seed"         : 42,
    "output_dir"   : f"{OUTPUT_PATH}output_indobert_ft_no_fe",
    "score_min"    : 2,
    "score_max"    : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {DEVICE}")
print(f"Model          : {CFG['model_name']}")
print(f"Fitur tambahan : TIDAK DIGUNAKAN (BERT only)")


# ══════════════════════════════════════════════════════════
#  REPRODUCIBILITY
# ══════════════════════════════════════════════════════════
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating, max_rating):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)
    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1
    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored
    weights    = np.array([[(i - j) ** 2 / (num_ratings - 1) ** 2
                             for j in range(num_ratings)]
                            for i in range(num_ratings)])
    denom = np.sum(weights * expected)
    return 0.0 if denom == 0 else 1.0 - np.sum(weights * conf_mat) / denom


# ══════════════════════════════════════════════════════════
#  DATASET (tanpa fitur linguistik)
# ══════════════════════════════════════════════════════════
class EssayDataset(Dataset):
    def __init__(self, texts, scores, tokenizer, max_len):
        self.texts     = texts
        self.scores    = scores
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True,
                             return_tensors="pt")
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "score"         : torch.tensor(self.scores[idx], dtype=torch.float32),
        }


# ══════════════════════════════════════════════════════════
#  MODEL — IndoBERTRegressor (BERT only, tanpa fitur tambahan)
# ══════════════════════════════════════════════════════════
class IndoBERTRegressor(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size  # 768
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, 256)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        return self.fc2(self.dropout(self.relu(self.fc1(cls)))).squeeze(-1)


# ══════════════════════════════════════════════════════════
#  TRAINING & EVALUATION
# ══════════════════════════════════════════════════════════
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        preds = model(batch["input_ids"].to(device),
                      batch["attention_mask"].to(device))
        loss  = criterion(preds, batch["score"].to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device, score_min, score_max):
    model.eval()
    all_preds, all_true, total_loss = [], [], 0.0
    criterion = nn.MSELoss()
    with torch.no_grad():
        for batch in loader:
            preds = model(batch["input_ids"].to(device),
                          batch["attention_mask"].to(device))
            total_loss += criterion(preds, batch["score"].to(device)).item()
            all_preds.extend(preds.cpu().numpy().tolist())
            all_true.extend(batch["score"].cpu().numpy().tolist())
    all_preds = np.array(all_preds)
    all_true  = np.array(all_true)
    return (total_loss / len(loader),
            mean_squared_error(all_true, all_preds),
            quadratic_weighted_kappa(all_true, all_preds, score_min, score_max),
            all_preds, all_true)


# ══════════════════════════════════════════════════════════
#  RUN FOLD
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx, texts,
             scores, question_ids, tokenizer, cfg, device, es_idx=None):

    print(f"\n{'='*55}\n  FOLD {fold_idx+1}/{cfg['n_folds']}\n{'='*55}")
    set_seed(cfg["seed"] + fold_idx)
    if device.type == "cuda":  # [FIX-MEM] true per-fold training peak
        import gc; gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(device)  # [FIX-LEAK]

    y_train = scores[train_idx]
    y_val   = scores[val_idx]

    train_ds = EssayDataset([texts[i] for i in train_idx], y_train,
                             tokenizer, cfg["max_len"])
    val_ds   = EssayDataset([texts[i] for i in val_idx],   y_val,
                             tokenizer, cfg["max_len"])
    train_dl = DataLoader(train_ds, batch_size=cfg["batch_size"],
                          shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                          shuffle=False, num_workers=0)
    # [FIX-LEAKAGE] early-stopping set (NOT the test fold)
    y_es     = scores[es_idx]
    es_ds   = EssayDataset([texts[i] for i in es_idx],   y_es,
                             tokenizer, cfg["max_len"])
    es_dl   = DataLoader(es_ds,   batch_size=cfg["batch_size"],
                          shuffle=False, num_workers=0)

    model = IndoBERTRegressor(cfg["model_name"], cfg["dropout"]).to(device)

    optimizer = torch.optim.AdamW([
        {"params": list(model.bert.parameters()), "lr": cfg["lr_encoder"]},
        {"params": list(model.fc1.parameters()) + list(model.fc2.parameters()),
         "lr": cfg["lr_head"]},
    ], weight_decay=cfg["weight_decay"])

    total_steps  = len(train_dl) * cfg["max_epochs"]
    warmup_steps = int(total_steps * cfg["warmup_ratio"])
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps)
    criterion = nn.MSELoss()

    best_val_loss, best_model_state, patience_counter = float("inf"), None, 0
    train_loss_history, val_loss_history = [], []
    val_qwk_history,   val_mse_history   = [], []
    n_epochs_actual = 0

    t_start = time.perf_counter()
    for epoch in range(cfg["max_epochs"]):
        n_epochs_actual = epoch + 1
        train_loss = train_one_epoch(model, train_dl, optimizer,
                                     scheduler, criterion, device)
        val_loss, val_mse, val_qwk, _, _ = evaluate(
            model, es_dl, device, cfg["score_min"], cfg["score_max"])

        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)
        val_qwk_history.append(val_qwk)
        val_mse_history.append(val_mse)

        print(f"  Epoch {epoch+1:02d}/{cfg['max_epochs']} | "
              f"train={train_loss:.4f} | val={val_loss:.4f} | "
              f"QWK={val_qwk:.4f} | MSE={val_mse:.4f}")

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg["patience"]:
                print(f"  Early stopping pada epoch {epoch+1}")
                break

    training_time = time.perf_counter() - t_start

    model.load_state_dict(best_model_state)

    # --- Inference time ---
    model.eval()
    sample_batch = next(iter(val_dl))
    N_INFER = min(100, len(val_ds))
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N_INFER):
            model(sample_batch["input_ids"][:1].to(device),
                  sample_batch["attention_mask"][:1].to(device))
    inference_ms = (time.perf_counter() - t0) / N_INFER * 1000

    # --- Memory tracking via pynvml (GPU only) ---
    if device.type == "cuda":
        pynvml.nvmlInit()
        handle   = pynvml.nvmlDeviceGetHandleByIndex(device.index or 0)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        mem_mb   = torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # [FIX-MEM] was whole-GPU "used"
        mem_device = "cuda"
        pynvml.nvmlShutdown()
    else:
        mem_mb     = 0.0
        mem_device = "cpu (not tracked)"

    _, final_mse, final_qwk, preds, true = evaluate(
        model, val_dl, device, cfg["score_min"], cfg["score_max"])

    fold_preds_df = pd.DataFrame({
        "sample_idx"  : val_idx,
        "question_id" : question_ids[val_idx],
        "true_score"  : true,
        "pred_score"  : preds,
        "residual"    : preds - true,
        "abs_error"   : np.abs(preds - true),
        "fold"        : fold_idx + 1,
    })

    print(f"\n  → Fold {fold_idx+1}: QWK={final_qwk:.4f} | MSE={final_mse:.4f} | "
          f"time={training_time:.1f}s | inf={inference_ms:.2f}ms | mem={mem_mb:.1f}MB")

    return {
        "fold"            : fold_idx + 1,
        "qwk"             : final_qwk,
        "mse"             : final_mse,
        "training_time_s" : training_time,
        "inference_ms"    : inference_ms,
        "memory_mb"       : mem_mb,
        "memory_device"   : mem_device,
        "best_val_loss"   : best_val_loss,
        "n_epochs_actual" : n_epochs_actual,
        "val_idx"         : val_idx.tolist(),
        "train_idx"       : train_idx.tolist(),
        "history": {
            "train_loss": train_loss_history,
            "val_loss"  : val_loss_history,
            "val_qwk"   : val_qwk_history,
            "val_mse"   : val_mse_history,
        },
    }, fold_preds_df


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    set_seed(CFG["seed"])

    print("\n[1/4] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])

    if os.path.exists(CFG["question_ids"]):
        question_ids = joblib.load(CFG["question_ids"])
    else:
        n_per_q = len(texts) // 10
        question_ids = np.repeat(np.arange(1, 11), n_per_q).astype(int)
        if len(question_ids) < len(texts):
            question_ids = np.append(question_ids,
                np.full(len(texts) - len(question_ids), 10, dtype=int))
        joblib.dump(question_ids, CFG["question_ids"])
        print(f"  question_ids dibuat dan disimpan (asumsi {n_per_q} esai/soal)")

    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())
    print(f"  Sampel: {len(texts)} | Skor: {CFG['score_min']}–{CFG['score_max']} | "
          f"Mean: {scores.mean():.2f}±{scores.std():.2f}")

    print(f"\n[2/4] Memuat tokenizer: {CFG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

    print(f"\n[3/4] Memulai {CFG['n_folds']}-fold CV (tanpa fitur linguistik)...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
    all_results, all_preds_dfs = [], []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(texts)):

        train_idx, es_idx = inner_split(train_idx, scores, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df = run_fold(
                fold_idx, train_idx, val_idx,
                texts, scores, question_ids,
                tokenizer, CFG, DEVICE, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)
        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1}: {e}")
            traceback.print_exc()

    print(f"\n[4/4] Menyimpan hasil...")
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"])
    overall_mse = mean_squared_error(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values)

    score_dist = {}
    for q in range(1, 11):
        mask = question_ids == q
        if mask.sum() > 0:
            score_dist[int(q)] = {
                "mean"  : float(scores[mask].mean()),
                "std"   : float(scores[mask].std()),
                "n"     : int(mask.sum()),
                "counts": {int(k): int(v) for k, v in
                           zip(*np.unique(scores[mask].astype(int),
                                          return_counts=True))},
            }

    experiment_data = {
        "config"            : CFG,
        "fold_results"      : all_results,
        "fold_predictions"  : all_preds_df,
        "overall_qwk"       : overall_qwk,
        "overall_mse"       : overall_mse,
        "scores"            : scores,
        "question_ids"      : question_ids,
        "score_distribution": score_dist,
        "model_name"        : CFG["model_name"],
        "paradigm"          : "fine-tuning (BERT only, no linguistic features)",
        "n_samples"         : len(texts),
        "timestamp"         : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    joblib_path = os.path.join(CFG["output_dir"], "experiment_results_no_fe.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df     = pd.DataFrame(all_results)
    results_df_csv = results_df.drop(columns=["history","val_idx","train_idx"],
                                      errors="ignore")
    results_df_csv.to_csv(
        os.path.join(CFG["output_dir"], "cv_results_indobert_no_fe.csv"), index=False)
    all_preds_df.to_csv(
        os.path.join(CFG["output_dir"], "all_predictions_indobert_no_fe.csv"), index=False)

    print(f"\n{'='*55}")
    print("  RINGKASAN — IndoBERT Fine-tuning (BERT ONLY, tanpa fitur linguistik)")
    print(f"{'='*55}")
    print(results_df_csv.to_string(index=False))
    print(f"\n  Rata-rata ± Std:")
    for m in ["qwk","mse","training_time_s","inference_ms","memory_mb"]:
        print(f"    {m:<20s}: {results_df[m].mean():.4f} ± {results_df[m].std():.4f}")
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")
    print(f"\n  Disimpan di     : {joblib_path}")

    return experiment_data


if __name__ == "__main__":
    main()


---


In [ ]:
"""
=============================================================
  AES — Paradigma A: DistilBERT-Indo Fine-tuning (BERT ONLY, tanpa fitur linguistik)
  Dataset  : preprocessed_texts.pkl, scores.pkl   (SAMA dengan versi +FE)
  Model    : cahya/distilbert-base-indonesian
  Arsitektur: DistilBERT 6L (from scratch, Indo) → [CLS] → 768
  Metrik   : QWK, MSE
  Evaluasi : 10-Fold Cross-Validation
=============================================================
"""

import os, time, traceback, random
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pynvml
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import KFold
# ── [FIX-LEAKAGE] inner early-stopping split ────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ES_FRAC        = 0.10   # share of the outer-train fold used for early stopping
SCALE_FEATURES = True   # FA-FT only: z-score linguistic features (fit on train only)

def inner_split(train_idx, y, fold_idx, frac=ES_FRAC, seed=42):
    """Split the outer-train fold into (train, early-stop) sets.
    Stratified by score band (scores <=5 merged) so rare low scores appear in both.
    Same seed in every notebook -> identical splits for FA-FT and FA-ML."""
    bands = np.clip(np.asarray(y)[train_idx].astype(int), 5, 10)
    tr, es = train_test_split(train_idx, test_size=frac,
                              random_state=seed + fold_idx, stratify=bands)
    return np.sort(tr), np.sort(es)
# ────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_squared_error


# ══════════════════════════════════════════════════════════
#  KONFIGURASI
# ══════════════════════════════════════════════════════════
CFG = {
    "essay"        : f"{ESSAY_PATH}preprocessed_texts.pkl",
    "score"        : f"{SCORE_PATH}scores.pkl",
    "question_ids" : f"{SCORE_PATH}question_ids.pkl",
    "model_name"   : "cahya/distilbert-base-indonesian",
    "max_len"      : 512,
    "lr_encoder"   : 3e-5,          # Sanh et al. (2019) untuk DistilBERT
    "lr_head"      : 1e-3,
    "weight_decay" : 0.01,
    "warmup_ratio" : 0.10,
    "max_epochs"   : 30,
    "batch_size"   : 64,
    "patience"     : 5,
    "dropout"      : 0.1,
    "n_folds"      : 10,
    "seed"         : 42,
    "output_dir"   : f"{OUTPUT_PATH}output_distilbert_ft_no_fe",
    "score_min"    : 2,
    "score_max"    : 10,
}

os.makedirs(CFG["output_dir"], exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device         : {DEVICE}")
print(f"Model          : {CFG['model_name']} (DistilBERT 6L, [CLS], hidden 768)")
print(f"Fitur tambahan : TIDAK DIGUNAKAN (BERT only)")


# ══════════════════════════════════════════════════════════
#  REPRODUCIBILITY
# ══════════════════════════════════════════════════════════
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ══════════════════════════════════════════════════════════
#  QWK
# ══════════════════════════════════════════════════════════
def quadratic_weighted_kappa(y_true, y_pred, min_rating, max_rating):
    y_true = np.array(y_true, dtype=int)
    y_pred = np.round(np.clip(y_pred, min_rating, max_rating)).astype(int)
    num_ratings = max_rating - min_rating + 1
    conf_mat    = np.zeros((num_ratings, num_ratings), dtype=int)
    for a, p in zip(y_true, y_pred):
        conf_mat[a - min_rating][p - min_rating] += 1
    num_scored = len(y_true)
    hist_true  = np.bincount(y_true - min_rating, minlength=num_ratings)
    hist_pred  = np.bincount(y_pred - min_rating, minlength=num_ratings)
    expected   = np.outer(hist_true, hist_pred) / num_scored
    weights    = np.array([[(i - j) ** 2 / (num_ratings - 1) ** 2
                             for j in range(num_ratings)]
                            for i in range(num_ratings)])
    denom = np.sum(weights * expected)
    return 0.0 if denom == 0 else 1.0 - np.sum(weights * conf_mat) / denom


# ══════════════════════════════════════════════════════════
#  DATASET (tanpa fitur linguistik)
# ══════════════════════════════════════════════════════════
class EssayDataset(Dataset):
    def __init__(self, texts, scores, tokenizer, max_len):
        self.texts     = texts
        self.scores    = scores
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                             padding="max_length", truncation=True,
                             return_tensors="pt")
        return {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "score"         : torch.tensor(self.scores[idx], dtype=torch.float32),
        }


# ══════════════════════════════════════════════════════════
#  MODEL — DistilBERTRegressor (BERT only, tanpa fitur tambahan)
# ══════════════════════════════════════════════════════════
class DistilBERTRegressor(nn.Module):
    def __init__(self, model_name, dropout=0.1):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size  # 768
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, 256)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        return self.fc2(self.dropout(self.relu(self.fc1(cls)))).squeeze(-1)


# ══════════════════════════════════════════════════════════
#  TRAINING & EVALUATION
# ══════════════════════════════════════════════════════════
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        optimizer.zero_grad()
        preds = model(batch["input_ids"].to(device),
                      batch["attention_mask"].to(device))
        loss  = criterion(preds, batch["score"].to(device))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device, score_min, score_max):
    model.eval()
    all_preds, all_true, total_loss = [], [], 0.0
    criterion = nn.MSELoss()
    with torch.no_grad():
        for batch in loader:
            preds = model(batch["input_ids"].to(device),
                          batch["attention_mask"].to(device))
            total_loss += criterion(preds, batch["score"].to(device)).item()
            all_preds.extend(preds.cpu().numpy().tolist())
            all_true.extend(batch["score"].cpu().numpy().tolist())
    all_preds = np.array(all_preds)
    all_true  = np.array(all_true)
    return (total_loss / len(loader),
            mean_squared_error(all_true, all_preds),
            quadratic_weighted_kappa(all_true, all_preds, score_min, score_max),
            all_preds, all_true)


# ══════════════════════════════════════════════════════════
#  RUN FOLD
# ══════════════════════════════════════════════════════════
def run_fold(fold_idx, train_idx, val_idx, texts,
             scores, question_ids, tokenizer, cfg, device, es_idx=None):

    print(f"\n{'='*55}\n  FOLD {fold_idx+1}/{cfg['n_folds']}\n{'='*55}")
    set_seed(cfg["seed"] + fold_idx)
    if device.type == "cuda":  # [FIX-MEM] true per-fold training peak
        import gc; gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(device)  # [FIX-LEAK]

    y_train = scores[train_idx]
    y_val   = scores[val_idx]

    train_ds = EssayDataset([texts[i] for i in train_idx], y_train,
                             tokenizer, cfg["max_len"])
    val_ds   = EssayDataset([texts[i] for i in val_idx],   y_val,
                             tokenizer, cfg["max_len"])
    train_dl = DataLoader(train_ds, batch_size=cfg["batch_size"],
                          shuffle=True,  num_workers=0)
    val_dl   = DataLoader(val_ds,   batch_size=cfg["batch_size"],
                          shuffle=False, num_workers=0)
    # [FIX-LEAKAGE] early-stopping set (NOT the test fold)
    y_es     = scores[es_idx]
    es_ds   = EssayDataset([texts[i] for i in es_idx],   y_es,
                             tokenizer, cfg["max_len"])
    es_dl   = DataLoader(es_ds,   batch_size=cfg["batch_size"],
                          shuffle=False, num_workers=0)

    model = DistilBERTRegressor(cfg["model_name"], cfg["dropout"]).to(device)

    optimizer = torch.optim.AdamW([
        {"params": list(model.bert.parameters()), "lr": cfg["lr_encoder"]},
        {"params": list(model.fc1.parameters()) + list(model.fc2.parameters()),
         "lr": cfg["lr_head"]},
    ], weight_decay=cfg["weight_decay"])

    total_steps  = len(train_dl) * cfg["max_epochs"]
    warmup_steps = int(total_steps * cfg["warmup_ratio"])
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps)
    criterion = nn.MSELoss()

    best_val_loss, best_model_state, patience_counter = float("inf"), None, 0
    train_loss_history, val_loss_history = [], []
    val_qwk_history,   val_mse_history   = [], []
    n_epochs_actual = 0

    t_start = time.perf_counter()
    for epoch in range(cfg["max_epochs"]):
        n_epochs_actual = epoch + 1
        train_loss = train_one_epoch(model, train_dl, optimizer,
                                     scheduler, criterion, device)
        val_loss, val_mse, val_qwk, _, _ = evaluate(
            model, es_dl, device, cfg["score_min"], cfg["score_max"])

        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)
        val_qwk_history.append(val_qwk)
        val_mse_history.append(val_mse)

        print(f"  Epoch {epoch+1:02d}/{cfg['max_epochs']} | "
              f"train={train_loss:.4f} | val={val_loss:.4f} | "
              f"QWK={val_qwk:.4f} | MSE={val_mse:.4f}")

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cfg["patience"]:
                print(f"  Early stopping pada epoch {epoch+1}")
                break

    training_time = time.perf_counter() - t_start

    model.load_state_dict(best_model_state)

    # --- Inference time ---
    model.eval()
    sample_batch = next(iter(val_dl))
    N_INFER = min(100, len(val_ds))
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(N_INFER):
            model(sample_batch["input_ids"][:1].to(device),
                  sample_batch["attention_mask"][:1].to(device))
    inference_ms = (time.perf_counter() - t0) / N_INFER * 1000

    # --- Memory tracking via pynvml (GPU only) ---
    if device.type == "cuda":
        pynvml.nvmlInit()
        handle   = pynvml.nvmlDeviceGetHandleByIndex(device.index or 0)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
        mem_mb   = torch.cuda.max_memory_allocated(device) / (1024 ** 2)  # [FIX-MEM] was whole-GPU "used"
        mem_device = "cuda"
        pynvml.nvmlShutdown()
    else:
        mem_mb     = 0.0
        mem_device = "cpu (not tracked)"

    _, final_mse, final_qwk, preds, true = evaluate(
        model, val_dl, device, cfg["score_min"], cfg["score_max"])

    fold_preds_df = pd.DataFrame({
        "sample_idx"  : val_idx,
        "question_id" : question_ids[val_idx],
        "true_score"  : true,
        "pred_score"  : preds,
        "residual"    : preds - true,
        "abs_error"   : np.abs(preds - true),
        "fold"        : fold_idx + 1,
    })

    print(f"\n  → Fold {fold_idx+1}: QWK={final_qwk:.4f} | MSE={final_mse:.4f} | "
          f"time={training_time:.1f}s | inf={inference_ms:.2f}ms | mem={mem_mb:.1f}MB")

    return {
        "fold"            : fold_idx + 1,
        "qwk"             : final_qwk,
        "mse"             : final_mse,
        "training_time_s" : training_time,
        "inference_ms"    : inference_ms,
        "memory_mb"       : mem_mb,
        "memory_device"   : mem_device,
        "best_val_loss"   : best_val_loss,
        "n_epochs_actual" : n_epochs_actual,
        "val_idx"         : val_idx.tolist(),
        "train_idx"       : train_idx.tolist(),
        "history": {
            "train_loss": train_loss_history,
            "val_loss"  : val_loss_history,
            "val_qwk"   : val_qwk_history,
            "val_mse"   : val_mse_history,
        },
    }, fold_preds_df


# ══════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════
def main():
    set_seed(CFG["seed"])

    print("\n[1/4] Membaca dataset...")
    texts  = joblib.load(CFG["essay"])
    scores = joblib.load(CFG["score"])

    if os.path.exists(CFG["question_ids"]):
        question_ids = joblib.load(CFG["question_ids"])
    else:
        n_per_q = len(texts) // 10
        question_ids = np.repeat(np.arange(1, 11), n_per_q).astype(int)
        if len(question_ids) < len(texts):
            question_ids = np.append(question_ids,
                np.full(len(texts) - len(question_ids), 10, dtype=int))
        joblib.dump(question_ids, CFG["question_ids"])
        print(f"  question_ids dibuat dan disimpan (asumsi {n_per_q} esai/soal)")

    CFG["score_min"] = int(scores.min())
    CFG["score_max"] = int(scores.max())
    print(f"  Sampel: {len(texts)} | Skor: {CFG['score_min']}–{CFG['score_max']} | "
          f"Mean: {scores.mean():.2f}±{scores.std():.2f}")

    print(f"\n[2/4] Memuat tokenizer: {CFG['model_name']}...")
    tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

    print(f"\n[3/4] Memulai {CFG['n_folds']}-fold CV (tanpa fitur linguistik)...")
    kf = KFold(n_splits=CFG["n_folds"], shuffle=True, random_state=CFG["seed"])
    all_results, all_preds_dfs = [], []

    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(texts)):

        train_idx, es_idx = inner_split(train_idx, scores, fold_idx)  # [FIX-LEAKAGE]
        try:
            result, preds_df = run_fold(
                fold_idx, train_idx, val_idx,
                texts, scores, question_ids,
                tokenizer, CFG, DEVICE, es_idx=es_idx)
            all_results.append(result)
            all_preds_dfs.append(preds_df)
        except Exception as e:
            print(f"  [ERROR] Fold {fold_idx+1}: {e}")
            traceback.print_exc()

    print(f"\n[4/4] Menyimpan hasil...")
    all_preds_df = pd.concat(all_preds_dfs, ignore_index=True)

    overall_qwk = quadratic_weighted_kappa(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values,
        CFG["score_min"], CFG["score_max"])
    overall_mse = mean_squared_error(
        all_preds_df["true_score"].values,
        all_preds_df["pred_score"].values)

    score_dist = {}
    for q in range(1, 11):
        mask = question_ids == q
        if mask.sum() > 0:
            score_dist[int(q)] = {
                "mean"  : float(scores[mask].mean()),
                "std"   : float(scores[mask].std()),
                "n"     : int(mask.sum()),
                "counts": {int(k): int(v) for k, v in
                           zip(*np.unique(scores[mask].astype(int),
                                          return_counts=True))},
            }

    experiment_data = {
        "config"            : CFG,
        "fold_results"      : all_results,
        "fold_predictions"  : all_preds_df,
        "overall_qwk"       : overall_qwk,
        "overall_mse"       : overall_mse,
        "scores"            : scores,
        "question_ids"      : question_ids,
        "score_distribution": score_dist,
        "model_name"        : CFG["model_name"],
        "paradigm"          : "fine-tuning (BERT only, no linguistic features)",
        "n_samples"         : len(texts),
        "timestamp"         : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    joblib_path = os.path.join(CFG["output_dir"], "experiment_results_no_fe.joblib")
    joblib.dump(experiment_data, joblib_path)

    results_df     = pd.DataFrame(all_results)
    results_df_csv = results_df.drop(columns=["history","val_idx","train_idx"],
                                      errors="ignore")
    results_df_csv.to_csv(
        os.path.join(CFG["output_dir"], "cv_results_distilbert_no_fe.csv"), index=False)
    all_preds_df.to_csv(
        os.path.join(CFG["output_dir"], "all_predictions_distilbert_no_fe.csv"), index=False)

    print(f"\n{'='*55}")
    print("  RINGKASAN — DistilBERT Fine-tuning (BERT ONLY, tanpa fitur linguistik)")
    print(f"{'='*55}")
    print(results_df_csv.to_string(index=False))
    print(f"\n  Rata-rata ± Std:")
    for m in ["qwk","mse","training_time_s","inference_ms","memory_mb"]:
        print(f"    {m:<20s}: {results_df[m].mean():.4f} ± {results_df[m].std():.4f}")
    print(f"\n  QWK keseluruhan : {overall_qwk:.4f}")
    print(f"  MSE keseluruhan : {overall_mse:.4f}")
    print(f"\n  Disimpan di     : {joblib_path}")

    return experiment_data


if __name__ == "__main__":
    main()
